## figure1a

In [ ]:
# ADM1 map: EDI_qmean basemap with sl_sevexp bubbles.

import math
import numpy as np
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, BoundaryNorm, ListedColormap

shp_path = r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
country_shp_path = r"../data/country/country_SSA_HE2.shp"

BASE_FIELD = "EDI_qmean"
BUBBLE_FIELD = "sl_sevexp"

BASE_LEGEND_TITLE = "EDI"
BUBBLE_LEGEND_TITLE = "SPW flood severity"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.size"] = 8

BASE_CMAP = ListedColormap(
    plt.get_cmap("Greys")(np.linspace(0.05, 0.60, 256))
)
BASE_EDGE_COLOR = "#ffffff"
COUNTRY_EDGE_COLOR = "black"

BUBBLE_CMAP = "RdYlBu_r"
BUBBLE_ALPHA = 0.7
BUBBLE_EDGE_COLOR = "white"

MIN_BUBBLE_SIZE = 20
MAX_BUBBLE_SIZE = 100

gdf = gpd.read_file(shp_path)
country_gdf = gpd.read_file(country_shp_path)

if country_gdf.crs != gdf.crs:
    country_gdf = country_gdf.to_crs(gdf.crs)

if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    country_gdf = country_gdf.to_crs(epsg=4326)

gdf = gdf.replace([np.inf, -np.inf], np.nan)

# Derive ADM1 IDs from ADM2 IDs, e.g. AGO.1.3_1 -> AGO.1_1.
gdf["GID_1"] = (
    gdf["GID_2"].astype(str)
    .str.extract(r"^([^.]+\.\d+)", expand=False)
    + "_1"
)

if gdf["GID_1"].isna().any():
    bad_ids = gdf.loc[gdf["GID_1"].isna(), "GID_2"].head().tolist()
    raise ValueError(f"Some GID_2 values cannot be converted to GID_1, e.g.: {bad_ids}")

# Compute ADM2 area in an equal-area CRS; do not use geographic-coordinate area.
agg_metric = gdf.to_crs("EPSG:6933").copy()

agg_metric.geometry = agg_metric.geometry.make_valid()
agg_metric["_area"] = agg_metric.geometry.area

# EDI_qmean uses ADM2 area weights; HE_perpop and flood use sl_pop weights.
agg_metric["_edi_weight"] = agg_metric["_area"].where(
    agg_metric[BASE_FIELD].notna(), 0
)
agg_metric["_edi_weighted"] = (
    agg_metric[BASE_FIELD].fillna(0) * agg_metric["_edi_weight"]
)

valid_he = agg_metric[BUBBLE_FIELD].notna() & (agg_metric["sl_pop"] > 0)
agg_metric["_he_weight"] = agg_metric["sl_pop"].where(valid_he, 0)
agg_metric["_he_weighted"] = (
    agg_metric[BUBBLE_FIELD].fillna(0) * agg_metric["_he_weight"]
)

adm1_metric = agg_metric.dissolve(
    by="GID_1",
    aggfunc={
        "_edi_weighted": "sum",
        "_edi_weight": "sum",
        "_he_weighted": "sum",
        "_he_weight": "sum",
    },
)

adm1_metric[BASE_FIELD] = (
    adm1_metric["_edi_weighted"] / adm1_metric["_edi_weight"]
).where(adm1_metric["_edi_weight"] > 0)

adm1_metric[BUBBLE_FIELD] = (
    adm1_metric["_he_weighted"] / adm1_metric["_he_weight"]
).where(adm1_metric["_he_weight"] > 0)

plot_gdf = adm1_metric.to_crs(epsg=4326)

bubble_gdf = plot_gdf.dropna(subset=[BUBBLE_FIELD]).copy()
bubble_gdf = bubble_gdf[bubble_gdf[BUBBLE_FIELD] > 0].copy()

metric_crs = "EPSG:3857"

bubble_metric = bubble_gdf.to_crs(metric_crs).copy()
bubble_metric["centroid_geom"] = bubble_metric.geometry.centroid

centroids = gpd.GeoDataFrame(
    bubble_metric.drop(columns="geometry"),
    geometry=bubble_metric["centroid_geom"],
    crs=metric_crs
).to_crs(plot_gdf.crs)

bubble_gdf["centroid_geom"] = centroids.geometry

if len(bubble_gdf) > 0:
# Bin positive values into five quantile classes; size and color use the same rank.
    bubble_values = bubble_gdf[BUBBLE_FIELD].to_numpy(dtype=float)
    bubble_bounds = np.quantile(bubble_values, [0, 0.2, 0.4, 0.6, 0.8, 1.0])

    if len(np.unique(bubble_bounds)) < 6:
        raise ValueError(
            f"{BUBBLE_FIELD} quantile boundaries contain duplicate values; cannot split into five stable classes."
        )

    bubble_gdf["bubble_class"] = np.digitize(
        bubble_values,
        bubble_bounds[1:-1],
        right=True,
    )

    bubble_class_sizes = np.linspace(MIN_BUBBLE_SIZE, MAX_BUBBLE_SIZE, 5)
    bubble_gdf["plot_size"] = bubble_class_sizes[
        bubble_gdf["bubble_class"].to_numpy(dtype=int)
    ]
else:
    bubble_bounds = np.array([])
    bubble_class_sizes = np.linspace(MIN_BUBBLE_SIZE, MAX_BUBBLE_SIZE, 5)
    bubble_gdf["bubble_class"] = np.nan
    bubble_gdf["plot_size"] = np.nan

bubble_colors = plt.get_cmap(BUBBLE_CMAP)(np.linspace(0.05, 0.95, 5))
bubble_cmap = ListedColormap(bubble_colors)
bubble_norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), bubble_cmap.N)

valid_base = plot_gdf.loc[plot_gdf[BASE_FIELD].notna(), BASE_FIELD]

if valid_base.empty:
    raise ValueError(f"{BASE_FIELD} is entirely missing; cannot draw the grayscale basemap.")

base_vmin = valid_base.min()
base_vmax = valid_base.max()

base_norm = Normalize(vmin=base_vmin, vmax=base_vmax)

fig, ax = plt.subplots(figsize=(6.8, 6.8), dpi=300)

plot_gdf.boundary.plot(
    ax=ax,
    linewidth=0.15,
    edgecolor=BASE_EDGE_COLOR,
    zorder=1
)

plot_gdf.plot(
    column=BASE_FIELD,
    ax=ax,
    cmap=BASE_CMAP,
    norm=base_norm,
    linewidth=0.15,
    edgecolor=BASE_EDGE_COLOR,
    legend=False,
    missing_kwds={
        "color": "#F2F2F2",
        "edgecolor": BASE_EDGE_COLOR
    },
    zorder=2
)

country_gdf.boundary.plot(
    ax=ax,
    linewidth=0.25,
    edgecolor=COUNTRY_EDGE_COLOR,
    facecolor="none",
    zorder=3
)

outer_boundary = country_gdf.dissolve()
outer_boundary.boundary.plot(
    ax=ax,
    linewidth=0.7,
    edgecolor="black",
    facecolor="none",
    zorder=4
)

if not bubble_gdf.empty:
    x = bubble_gdf["centroid_geom"].x
    y = bubble_gdf["centroid_geom"].y

    ax.scatter(
        x, y,
        s=bubble_gdf["plot_size"],
        c=bubble_gdf["bubble_class"],
        cmap=bubble_cmap,
        norm=bubble_norm,
        alpha=BUBBLE_ALPHA,
        edgecolor=BUBBLE_EDGE_COLOR,
        linewidth=0.3,
        zorder=5
    )

ax.set_xlim(-20, 55)
ax.set_ylim(-40, 32)

ax.set_xticks([])
ax.set_yticks([])

# Explicitly hide the axis frame.
ax.set_frame_on(False)
ax.patch.set_visible(False)
for spine in ax.spines.values():
    spine.set_visible(False)

def add_north_arrow(ax, x=0.86, y=0.20, size=0.045):
    triangle = np.array([
        [x, y + size * 0.70],
        [x - size * 0.32, y - size * 0.35],
        [x + size * 0.32, y - size * 0.35]
    ])

    arrow = Polygon(
        triangle,
        closed=True,
        facecolor="black",
        edgecolor="black",
        transform=ax.transAxes,
        zorder=10,
        clip_on=False
    )
    ax.add_patch(arrow)

    ax.text(
        x,
        y - size * 0.55,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=12,
        fontweight="bold",
        color="black",
        zorder=11
    )

def add_scalebar(ax, length_km=1000, location=(0.75, 0.08), linewidth=3):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    x0 = xmin + location[0] * (xmax - xmin)
    y0 = ymin + location[1] * (ymax - ymin)

    lat0 = y0
    km_per_deg_lon = 111.32 * math.cos(math.radians(lat0))
    if km_per_deg_lon <= 0:
        km_per_deg_lon = 111.32

    length_deg = length_km / km_per_deg_lon
    x1 = x0 + length_deg

    if x1 > xmax - 0.03 * (xmax - xmin):
        x1 = xmax - 0.03 * (xmax - xmin)
        x0 = x1 - length_deg

    ax.plot(
        [x0, x1],
        [y0, y0],
        color="black",
        linewidth=linewidth,
        solid_capstyle="butt",
        zorder=10
    )

    ax.text(
        (x0 + x1) / 2,
        y0 + 1.1,
        f"{length_km} km",
        ha="center",
        va="bottom",
        fontsize=10,
        color="black",
        zorder=11
    )

add_north_arrow(ax, x=0.81, y=0.12, size=0.02)
add_scalebar(ax, length_km=1000, location=(0.74, 0.02), linewidth=2)

sm = ScalarMappable(norm=base_norm, cmap=BASE_CMAP)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    orientation="vertical",
    fraction=0.018,
    pad=0.015,
    shrink=0.38,
    aspect=24
)
cbar.set_label(BASE_LEGEND_TITLE, fontsize=10, labelpad=5)
cbar.ax.tick_params(labelsize=10, length=2, width=0.5)
cbar.outline.set_linewidth(0.5)

def format_num(x):
    if x == 0:
        return "0"
    elif x >= 1_000_000:
        return f"{x:,.0f}"
    elif x >= 1_000:
        return f"{x:,.0f}"
    elif x >= 100:
        return f"{x:.0f}"
    elif x >= 10:
        return f"{x:.1f}"
    elif x >= 1:
        return f"{x:.2f}"
    else:
        return f"{x:.6f}"

if len(bubble_bounds) == 6:
    size_handles = []
    for i in range(5):
        lower = bubble_bounds[i]
        upper = bubble_bounds[i + 1]
        left_bracket = "[" if i == 0 else "("
        class_label = (
            f"{left_bracket}{format_num(lower)}, {format_num(upper)}]"
        )
        size_handles.append(
            plt.scatter(
                [], [],
                s=bubble_class_sizes[i],
                color=bubble_colors[i],
                edgecolor="white",
                linewidth=0.5,
                alpha=BUBBLE_ALPHA,
                label=class_label,
            )
        )

    legend2 = ax.legend(
        handles=size_handles,
        title=BUBBLE_LEGEND_TITLE,
        loc="lower left",
        frameon=False,
        fontsize=10,
        title_fontsize=10,
        labelspacing=1.0,
        borderpad=0.2,
        scatterpoints=1
    )
    ax.add_artist(legend2)

plt.tight_layout()

plt.savefig(
    "figure1a.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

## figure1b

In [ ]:
# ADM1 map: EDI_qmean basemap with HE_perpop bubbles.

import math
import numpy as np
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, BoundaryNorm, ListedColormap

shp_path = r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
country_shp_path = r"../data/country/country_SSA_HE2.shp"

BASE_FIELD = "EDI_qmean"
BUBBLE_FIELD = "HE_perpop"

BASE_LEGEND_TITLE = "EDI"
BUBBLE_LEGEND_TITLE = "SPW extreme heat exposure"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.size"] = 8

BASE_CMAP = ListedColormap(
    plt.get_cmap("Greys")(np.linspace(0.05, 0.60, 256))
)
BASE_EDGE_COLOR = "#ffffff"
COUNTRY_EDGE_COLOR = "black"

BUBBLE_CMAP = "RdYlBu_r"
BUBBLE_ALPHA = 0.7
BUBBLE_EDGE_COLOR = "white"

MIN_BUBBLE_SIZE = 20
MAX_BUBBLE_SIZE = 100

gdf = gpd.read_file(shp_path)
country_gdf = gpd.read_file(country_shp_path)

if country_gdf.crs != gdf.crs:
    country_gdf = country_gdf.to_crs(gdf.crs)

if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    country_gdf = country_gdf.to_crs(epsg=4326)

gdf = gdf.replace([np.inf, -np.inf], np.nan)

# Derive ADM1 IDs from ADM2 IDs, e.g. AGO.1.3_1 -> AGO.1_1.
gdf["GID_1"] = (
    gdf["GID_2"].astype(str)
    .str.extract(r"^([^.]+\.\d+)", expand=False)
    + "_1"
)

if gdf["GID_1"].isna().any():
    bad_ids = gdf.loc[gdf["GID_1"].isna(), "GID_2"].head().tolist()
    raise ValueError(f"Some GID_2 values cannot be converted to GID_1, e.g.: {bad_ids}")

# Compute ADM2 area in an equal-area CRS; do not use geographic-coordinate area.
agg_metric = gdf.to_crs("EPSG:6933").copy()

agg_metric.geometry = agg_metric.geometry.make_valid()
agg_metric["_area"] = agg_metric.geometry.area

# EDI_qmean uses ADM2 area weights; HE_perpop and flood use sl_pop weights.
agg_metric["_edi_weight"] = agg_metric["_area"].where(
    agg_metric[BASE_FIELD].notna(), 0
)
agg_metric["_edi_weighted"] = (
    agg_metric[BASE_FIELD].fillna(0) * agg_metric["_edi_weight"]
)

valid_he = agg_metric[BUBBLE_FIELD].notna() & (agg_metric["sl_pop"] > 0)
agg_metric["_he_weight"] = agg_metric["sl_pop"].where(valid_he, 0)
agg_metric["_he_weighted"] = (
    agg_metric[BUBBLE_FIELD].fillna(0) * agg_metric["_he_weight"]
)

adm1_metric = agg_metric.dissolve(
    by="GID_1",
    aggfunc={
        "_edi_weighted": "sum",
        "_edi_weight": "sum",
        "_he_weighted": "sum",
        "_he_weight": "sum",
    },
)

adm1_metric[BASE_FIELD] = (
    adm1_metric["_edi_weighted"] / adm1_metric["_edi_weight"]
).where(adm1_metric["_edi_weight"] > 0)

adm1_metric[BUBBLE_FIELD] = (
    adm1_metric["_he_weighted"] / adm1_metric["_he_weight"]
).where(adm1_metric["_he_weight"] > 0)

plot_gdf = adm1_metric.to_crs(epsg=4326)

bubble_gdf = plot_gdf.dropna(subset=[BUBBLE_FIELD]).copy()
bubble_gdf = bubble_gdf[bubble_gdf[BUBBLE_FIELD] > 0].copy()

metric_crs = "EPSG:3857"

bubble_metric = bubble_gdf.to_crs(metric_crs).copy()
bubble_metric["centroid_geom"] = bubble_metric.geometry.centroid

centroids = gpd.GeoDataFrame(
    bubble_metric.drop(columns="geometry"),
    geometry=bubble_metric["centroid_geom"],
    crs=metric_crs
).to_crs(plot_gdf.crs)

bubble_gdf["centroid_geom"] = centroids.geometry

if len(bubble_gdf) > 0:
# Bin positive values into five quantile classes; size and color use the same rank.
    bubble_values = bubble_gdf[BUBBLE_FIELD].to_numpy(dtype=float)
    bubble_bounds = np.quantile(bubble_values, [0, 0.2, 0.4, 0.6, 0.8, 1.0])

    if len(np.unique(bubble_bounds)) < 6:
        raise ValueError(
            f"{BUBBLE_FIELD} quantile boundaries contain duplicate values; cannot split into five stable classes."
        )

    bubble_gdf["bubble_class"] = np.digitize(
        bubble_values,
        bubble_bounds[1:-1],
        right=True,
    )

    bubble_class_sizes = np.linspace(MIN_BUBBLE_SIZE, MAX_BUBBLE_SIZE, 5)
    bubble_gdf["plot_size"] = bubble_class_sizes[
        bubble_gdf["bubble_class"].to_numpy(dtype=int)
    ]
else:
    bubble_bounds = np.array([])
    bubble_class_sizes = np.linspace(MIN_BUBBLE_SIZE, MAX_BUBBLE_SIZE, 5)
    bubble_gdf["bubble_class"] = np.nan
    bubble_gdf["plot_size"] = np.nan

bubble_colors = plt.get_cmap(BUBBLE_CMAP)(np.linspace(0.05, 0.95, 5))
bubble_cmap = ListedColormap(bubble_colors)
bubble_norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), bubble_cmap.N)

valid_base = plot_gdf.loc[plot_gdf[BASE_FIELD].notna(), BASE_FIELD]

if valid_base.empty:
    raise ValueError(f"{BASE_FIELD} is entirely missing; cannot draw the grayscale basemap.")

base_vmin = valid_base.min()
base_vmax = valid_base.max()

base_norm = Normalize(vmin=base_vmin, vmax=base_vmax)

fig, ax = plt.subplots(figsize=(6.8, 6.8), dpi=300)

plot_gdf.boundary.plot(
    ax=ax,
    linewidth=0.15,
    edgecolor=BASE_EDGE_COLOR,
    zorder=1
)

plot_gdf.plot(
    column=BASE_FIELD,
    ax=ax,
    cmap=BASE_CMAP,
    norm=base_norm,
    linewidth=0.15,
    edgecolor=BASE_EDGE_COLOR,
    legend=False,
    missing_kwds={
        "color": "#F2F2F2",
        "edgecolor": BASE_EDGE_COLOR
    },
    zorder=2
)

country_gdf.boundary.plot(
    ax=ax,
    linewidth=0.25,
    edgecolor=COUNTRY_EDGE_COLOR,
    facecolor="none",
    zorder=3
)

outer_boundary = country_gdf.dissolve()
outer_boundary.boundary.plot(
    ax=ax,
    linewidth=0.7,
    edgecolor="black",
    facecolor="none",
    zorder=4
)

if not bubble_gdf.empty:
    x = bubble_gdf["centroid_geom"].x
    y = bubble_gdf["centroid_geom"].y

    ax.scatter(
        x, y,
        s=bubble_gdf["plot_size"],
        c=bubble_gdf["bubble_class"],
        cmap=bubble_cmap,
        norm=bubble_norm,
        alpha=BUBBLE_ALPHA,
        edgecolor=BUBBLE_EDGE_COLOR,
        linewidth=0.3,
        zorder=5
    )

ax.set_xlim(-20, 55)
ax.set_ylim(-40, 32)

ax.set_xticks([])
ax.set_yticks([])

# Explicitly hide the axis frame.
ax.set_frame_on(False)
ax.patch.set_visible(False)
for spine in ax.spines.values():
    spine.set_visible(False)

def add_north_arrow(ax, x=0.86, y=0.20, size=0.045):
    triangle = np.array([
        [x, y + size * 0.70],
        [x - size * 0.32, y - size * 0.35],
        [x + size * 0.32, y - size * 0.35]
    ])

    arrow = Polygon(
        triangle,
        closed=True,
        facecolor="black",
        edgecolor="black",
        transform=ax.transAxes,
        zorder=10,
        clip_on=False
    )
    ax.add_patch(arrow)

    ax.text(
        x,
        y - size * 0.55,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=12,
        fontweight="bold",
        color="black",
        zorder=11
    )

def add_scalebar(ax, length_km=1000, location=(0.75, 0.08), linewidth=3):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    x0 = xmin + location[0] * (xmax - xmin)
    y0 = ymin + location[1] * (ymax - ymin)

    lat0 = y0
    km_per_deg_lon = 111.32 * math.cos(math.radians(lat0))
    if km_per_deg_lon <= 0:
        km_per_deg_lon = 111.32

    length_deg = length_km / km_per_deg_lon
    x1 = x0 + length_deg

    if x1 > xmax - 0.03 * (xmax - xmin):
        x1 = xmax - 0.03 * (xmax - xmin)
        x0 = x1 - length_deg

    ax.plot(
        [x0, x1],
        [y0, y0],
        color="black",
        linewidth=linewidth,
        solid_capstyle="butt",
        zorder=10
    )

    ax.text(
        (x0 + x1) / 2,
        y0 + 1.1,
        f"{length_km} km",
        ha="center",
        va="bottom",
        fontsize=10,
        color="black",
        zorder=11
    )

add_north_arrow(ax, x=0.81, y=0.12, size=0.02)
add_scalebar(ax, length_km=1000, location=(0.74, 0.02), linewidth=2)

sm = ScalarMappable(norm=base_norm, cmap=BASE_CMAP)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    orientation="vertical",
    fraction=0.018,
    pad=0.015,
    shrink=0.38,
    aspect=24
)
cbar.set_label(BASE_LEGEND_TITLE, fontsize=10, labelpad=5)
cbar.ax.tick_params(labelsize=10, length=2, width=0.5)
cbar.outline.set_linewidth(0.5)

def format_num(x):
    if x == 0:
        return "0"
    elif x >= 1_000_000:
        return f"{x:,.0f}"
    elif x >= 1_000:
        return f"{x:,.0f}"
    elif x >= 100:
        return f"{x:.0f}"
    elif x >= 10:
        return f"{x:.1f}"
    elif x >= 1:
        return f"{x:.2f}"
    else:
        return f"{x:.6f}"

if len(bubble_bounds) == 6:
    size_handles = []
    for i in range(5):
        lower = bubble_bounds[i]
        upper = bubble_bounds[i + 1]
        left_bracket = "[" if i == 0 else "("
        class_label = (
            f"{left_bracket}{format_num(lower)}, {format_num(upper)}]"
        )
        size_handles.append(
            plt.scatter(
                [], [],
                s=bubble_class_sizes[i],
                color=bubble_colors[i],
                edgecolor="white",
                linewidth=0.5,
                alpha=BUBBLE_ALPHA,
                label=class_label,
            )
        )

    legend2 = ax.legend(
        handles=size_handles,
        title=BUBBLE_LEGEND_TITLE,
        loc="lower left",
        frameon=False,
        fontsize=10,
        title_fontsize=10,
        labelspacing=1.0,
        borderpad=0.2,
        scatterpoints=1
    )
    ax.add_artist(legend2)

plt.tight_layout()

plt.savefig(
    "figure1b.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

## figure1d

In [ ]:
# ============================================================
# ADM1 malaria-rate aggregation
# sl_pfpr = sum(sl_pfinf) / sum(sl_pfpop)
# No plotting
# ============================================================

import math
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm, ListedColormap, Normalize
from matplotlib.patches import Polygon


# ============================================================
# 1. File path
# ============================================================

shp_path = (
    r"../data/overall/"
    r"admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
)
country_shp_path = r"../data/country/country_SSA_HE2.shp"


# ============================================================
# 2. ADM1 aggregation function
# ============================================================

def aggregate_adm1_malaria_rate(
    rate_field,
    numerator_field,
    denominator_field,
    base_field="EDI_qmean",
):
    """
    Aggregate ADM2 data to ADM1.

    Parameters
    ----------
    rate_field : str
        Name of the output malaria-rate field.

    numerator_field : str
        Numerator used to calculate malaria rate.

    denominator_field : str
        Denominator used to calculate malaria rate.

    base_field : str, default "EDI_qmean"
        Field aggregated to ADM1 using ADM2 area-weighted mean.

    Returns
    -------
    GeoDataFrame
        ADM1 geometries with area-weighted EDI and malaria rate.
    """

    # --------------------------------------------------------
    # Read ADM2 data
    # --------------------------------------------------------

    gdf = (
        gpd.read_file(shp_path)
        .replace([np.inf, -np.inf], np.nan)
    )


    # --------------------------------------------------------
    # Construct GID_1 from GID_2
    # --------------------------------------------------------

    gdf["GID_1"] = (
        gdf["GID_2"]
        .astype(str)
        .str.extract(
            r"^([^.]+\.\d+)",
            expand=False
        )
        + "_1"
    )

    if gdf["GID_1"].isna().any():
        raise ValueError(
            "Some GID_2 values cannot be converted to GID_1."
        )


    # --------------------------------------------------------
    # Equal-area CRS
    # --------------------------------------------------------

    metric = gdf.to_crs("EPSG:6933").copy()

    metric.geometry = metric.geometry.make_valid()

    metric["_area"] = metric.geometry.area


    # --------------------------------------------------------
    # Area-weighted ADM1 EDI
    # --------------------------------------------------------

    metric["_edi_weight"] = metric["_area"].where(
        metric[base_field].notna(),
        0
    )

    metric["_edi_weighted"] = (
        metric[base_field].fillna(0)
        * metric["_edi_weight"]
    )


    # --------------------------------------------------------
    # Malaria rate components
    # --------------------------------------------------------

    valid_rate = (
        metric[numerator_field].notna()
        & metric[denominator_field].notna()
        & (metric[denominator_field] >= 0)
    )

    metric["_rate_num"] = (
        metric[numerator_field]
        .where(valid_rate, 0)
    )

    metric["_rate_den"] = (
        metric[denominator_field]
        .where(valid_rate, 0)
    )

    metric["_rate_count"] = (
        valid_rate.astype(int)
    )


    # --------------------------------------------------------
    # Dissolve ADM2 -> ADM1
    # --------------------------------------------------------

    adm1 = metric.dissolve(
        by="GID_1",
        aggfunc={
            "_edi_weighted": "sum",
            "_edi_weight": "sum",
            "_rate_num": "sum",
            "_rate_den": "sum",
            "_rate_count": "sum",
        },
    )


    # --------------------------------------------------------
    # ADM1 area-weighted EDI
    # --------------------------------------------------------

    adm1[base_field] = (
        adm1["_edi_weighted"]
        / adm1["_edi_weight"]
    ).where(
        adm1["_edi_weight"] > 0
    )


    # --------------------------------------------------------
    # ADM1 malaria rate
    # --------------------------------------------------------

    adm1[rate_field] = (
        adm1["_rate_num"]
        / adm1["_rate_den"]
    ).where(
        (adm1["_rate_den"] > 0)
        & (adm1["_rate_count"] > 0)
    )


    # --------------------------------------------------------
    # Keep only necessary fields
    # --------------------------------------------------------

    adm1 = adm1[
        [
            base_field,
            rate_field,
            "_rate_num",
            "_rate_den",
            "geometry",
        ]
    ].copy()


    # Rename aggregated numerator / denominator for clarity
    adm1 = adm1.rename(
        columns={
            "_rate_num": f"{numerator_field}_sum",
            "_rate_den": f"{denominator_field}_sum",
        }
    )


    return adm1


# ============================================================
# 3. Calculate ADM1 malaria parasite rate
# ============================================================

adm1_pfpr = aggregate_adm1_malaria_rate(
    rate_field="sl_pfpr",
    numerator_field="sl_pfinf",
    denominator_field="sl_pfpop",
)

In [ ]:
# ADM1 malaria transition ratio maps.

def plot_adm1_malaria_count_ratio(
    ratio_field,
    numerator_field,
    denominator_field,
    bubble_legend_title,
):
    """Aggregate ADM1 counts first, then plot sum(numerator) / sum(denominator)."""
    # Reuse the ADM1 aggregation function from the previous cell.
    ratio_data = aggregate_adm1_malaria_rate(
        ratio_field, numerator_field, denominator_field
    )

    # Sum numerator and denominator independently before computing the ratio.
    raw = gpd.read_file(shp_path).replace([np.inf, -np.inf], np.nan)
    raw["GID_1"] = (
        raw["GID_2"].astype(str)
        .str.extract(r"^([^.]+\.\d+)", expand=False)
        + "_1"
    )
    count_sums = raw.groupby("GID_1")[[
        numerator_field, denominator_field
    ]].sum(min_count=1)

    rates = ratio_data[["EDI_qmean"]].join(count_sums, how="left")
    rates[ratio_field] = (
        rates[numerator_field] / rates[denominator_field]
    ).where(rates[denominator_field] > 0)
    rates["conversion_ratio"] = rates[ratio_field]

    # Build ADM1 geometries and join the transition-ratio results.
    geom = gpd.read_file(shp_path)
    geom["GID_1"] = (
        geom["GID_2"].astype(str)
        .str.extract(r"^([^.]+\.\d+)", expand=False)
        + "_1"
    )
    geom = geom.to_crs("EPSG:6933")
    geom.geometry = geom.geometry.make_valid()
    plot_gdf = geom[["GID_1", "geometry"]].dissolve(by="GID_1")
    plot_gdf = plot_gdf.join(rates).to_crs(epsg=4326)

    country_gdf = gpd.read_file(country_shp_path)
    if country_gdf.crs is not None and country_gdf.crs.to_epsg() != 4326:
        country_gdf = country_gdf.to_crs(epsg=4326)

    bubble_gdf = plot_gdf.dropna(subset=["conversion_ratio"]).copy()
    bubble_gdf = bubble_gdf[bubble_gdf["conversion_ratio"] > 0].copy()
    bubble_metric = bubble_gdf.to_crs("EPSG:6933").copy()
    bubble_metric["centroid_geom"] = bubble_metric.geometry.centroid
    centroids = gpd.GeoDataFrame(
        bubble_metric.drop(columns="geometry"),
        geometry=bubble_metric["centroid_geom"],
        crs="EPSG:6933",
    ).to_crs(epsg=4326)
    bubble_gdf["centroid_geom"] = centroids.geometry

    # Bin transition ratios into five quantile classes; values increase from blue to red.
    ratio_values = bubble_gdf["conversion_ratio"].to_numpy(dtype=float)
    ratio_bounds = np.quantile(ratio_values, [0, 0.2, 0.4, 0.6, 0.8, 1])
    if len(np.unique(ratio_bounds)) < 6:
        raise ValueError("The transition-ratio quantile boundaries contain duplicate values.")
    bubble_gdf["bubble_class"] = np.digitize(
        ratio_values, ratio_bounds[1:-1], right=True
    )
    bubble_sizes = np.linspace(20, 100, 5)
    bubble_gdf["plot_size"] = bubble_sizes[
        bubble_gdf["bubble_class"].to_numpy(dtype=int)
    ]
    bubble_colors = plt.get_cmap("BrBG_r")(np.linspace(0.05, 0.95, 5))
    bubble_cmap = ListedColormap(bubble_colors)
    bubble_norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), bubble_cmap.N)

    base_cmap = ListedColormap(
        plt.get_cmap("Greys")(np.linspace(0.05, 0.60, 256))
    )
    valid_base = plot_gdf["EDI_qmean"].dropna()
    base_norm = Normalize(vmin=valid_base.min(), vmax=valid_base.max())

    fig, ax = plt.subplots(figsize=(6.8, 6.8), dpi=300)
    plot_gdf.plot(
        column="EDI_qmean", ax=ax, cmap=base_cmap, norm=base_norm,
        linewidth=0.15, edgecolor="white", legend=False,
        missing_kwds={"color": "#F2F2F2", "edgecolor": "white"},
        zorder=2,
    )
    country_gdf.boundary.plot(
        ax=ax, linewidth=0.25, edgecolor="black", zorder=3
    )
    country_gdf.dissolve().boundary.plot(
        ax=ax, linewidth=0.7, edgecolor="black", zorder=4
    )
    ax.scatter(
        bubble_gdf["centroid_geom"].x,
        bubble_gdf["centroid_geom"].y,
        s=bubble_gdf["plot_size"],
        c=bubble_gdf["bubble_class"],
        cmap=bubble_cmap,
        norm=bubble_norm,
        alpha=0.78,
        edgecolor="white",
        linewidth=0.35,
        zorder=5,
    )

    ax.set_xlim(-20, 55)
    ax.set_ylim(-40, 32)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)
    ax.patch.set_visible(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    x_n, y_n, size_n = 0.81, 0.12, 0.02
    triangle = np.array([
        [x_n, y_n + size_n * 0.70],
        [x_n - size_n * 0.32, y_n - size_n * 0.35],
        [x_n + size_n * 0.32, y_n - size_n * 0.35],
    ])
    ax.add_patch(Polygon(
        triangle, closed=True, facecolor="black", edgecolor="black",
        transform=ax.transAxes, zorder=10, clip_on=False,
    ))
    ax.text(
        x_n, y_n - size_n * 0.55, "N", transform=ax.transAxes,
        ha="center", va="top", fontsize=12, fontweight="bold", zorder=11,
    )

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    x0 = xmin + 0.74 * (xmax - xmin)
    y0 = ymin + 0.02 * (ymax - ymin)
    length_deg = 1000 / (111.32 * math.cos(math.radians(y0)))
    x1 = x0 + length_deg
    ax.plot([x0, x1], [y0, y0], color="black", linewidth=2, zorder=10)
    ax.text(
        (x0 + x1) / 2, y0 + 1.1, "1000 km",
        ha="center", va="bottom", fontsize=10, zorder=11,
    )

    sm = ScalarMappable(norm=base_norm, cmap=base_cmap)
    sm.set_array([])
    cbar = fig.colorbar(
        sm, ax=ax, orientation="vertical", fraction=0.018,
        pad=0.015, shrink=0.38, aspect=24,
    )
    cbar.set_label("EDI", fontsize=10, labelpad=5)
    cbar.ax.tick_params(labelsize=10, length=2, width=0.5)
    cbar.outline.set_linewidth(0.5)

    def format_ratio(value):
        if value >= 100:
            return f"{value:.0f}"
        if value >= 10:
            return f"{value:.1f}"
        if value >= 1:
            return f"{value:.2f}"
        return f"{value:.4f}"

    handles = []
    for i in range(5):
        left = "[" if i == 0 else "("
        label = (
            f"{left}{format_ratio(ratio_bounds[i])}, "
            f"{format_ratio(ratio_bounds[i + 1])}]"
        )
        handles.append(plt.scatter(
            [], [], s=bubble_sizes[i], color=bubble_colors[i],
            edgecolor="white", linewidth=0.5, alpha=0.78,
            label=label,
        ))
    ax.legend(
        handles=handles, title=bubble_legend_title, loc="lower left",
        frameon=False, fontsize=10, title_fontsize=10,
        labelspacing=1.0, borderpad=0.2, scatterpoints=1,
    )

    result = rates[[
        "EDI_qmean", numerator_field, denominator_field,
        ratio_field, "conversion_ratio",
    ]].copy()
    return result

adm1_incidence_infection_ratio = plot_adm1_malaria_count_ratio(
    ratio_field="inc_inf_ratio",
    numerator_field="sl_pfinc",
    denominator_field="sl_pfinf",
    bubble_legend_title="Incidence-to-infection\nburden ratio",
)

plt.tight_layout()
plt.savefig(
        "figure1d.pdf",
        dpi=600,
        bbox_inches="tight"
    )
plt.show()

## figure1e

In [ ]:
# Mortality / incidence = sum(sl_pfmort) / sum(sl_pfinc).

adm1_mortality_incidence_ratio = plot_adm1_malaria_count_ratio(
    ratio_field="mort_inc_ratio",
    numerator_field="sl_pfmort",
    denominator_field="sl_pfinc",
    bubble_legend_title="Mortality-to-incidence\nburden ratio",
)

plt.tight_layout()
plt.savefig(
        "figure1e.pdf",
        dpi=600,
        bbox_inches="tight"
    )
plt.show()

## figure1c

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl

shp_path = r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
country_shp_path = r"../data/country/country_SSA_HE2.shp"

adm2 = gpd.read_file(shp_path).replace([np.inf, -np.inf], np.nan)
country_boundary = gpd.read_file(country_shp_path)

adm2["country_id"] = (
    pd.to_numeric(adm2["adm2ID"], errors="coerce") // 10000
).astype("Int64").astype(str)
country_boundary["country_id"] = country_boundary["country_id"].astype(str)

metric = adm2.to_crs("EPSG:6933").copy()
metric.geometry = metric.geometry.make_valid()
metric["_area"] = metric.geometry.area

def add_weighted_terms(frame, value, weight, prefix):
    valid = frame[value].notna() & frame[weight].notna() & (frame[weight] > 0)
    frame[f"_{prefix}_num"] = frame[value].where(valid, 0) * frame[weight].where(valid, 0)
    frame[f"_{prefix}_den"] = frame[weight].where(valid, 0)

add_weighted_terms(metric, "EDI_qmean", "_area", "edi")
add_weighted_terms(metric, "HE_perpop", "sl_pop", "heat")
add_weighted_terms(metric, "sl_sevexp", "sl_pop", "flood")

sum_fields = [
    "_edi_num", "_edi_den", "_heat_num", "_heat_den",
    "_flood_num", "_flood_den", "sl_pop", "sl_pfpop",
    "sl_pfinf", "sl_pfinc", "sl_pfmort",
]
country_stats = metric.groupby("country_id")[sum_fields].sum(min_count=1)
country_stats["EDI_qmean"] = country_stats["_edi_num"] / country_stats["_edi_den"]
country_stats["HE_perpop"] = country_stats["_heat_num"] / country_stats["_heat_den"]
country_stats["sl_sevexp"] = country_stats["_flood_num"] / country_stats["_flood_den"]
country_stats["sl_pfpr"] = country_stats["sl_pfinf"] / country_stats["sl_pfpop"]
country_stats["inc_inf_ratio"] = country_stats["sl_pfinc"] / country_stats["sl_pfinf"]
country_stats["mort_inc_ratio"] = country_stats["sl_pfmort"] / country_stats["sl_pfinc"]
country_stats = country_stats.replace([np.inf, -np.inf], np.nan)

country_gdf = country_boundary[["country_id", "country_na", "region", "geometry"]].merge(
    country_stats.reset_index(), on="country_id", how="left", validate="one_to_one"
)
if country_gdf.crs is not None and country_gdf.crs.to_epsg() != 4326:
    country_gdf = country_gdf.to_crs(4326)

map_specs = [
    ("EDI_qmean", "Environmental deprivation index", "Greys"),
    ("HE_perpop", "Slum population-weighted heat exposure", "YlOrRd"),
    ("sl_sevexp", "Slum population-weighted flood severity", "Blues"),
    ("sl_pfpr", "Malaria parasite rate", "YlGn"),
    ("inc_inf_ratio", "Incidence / infection ratio", "Oranges"),
    ("mort_inc_ratio", "Mortality / incidence ratio", "Purples"),
]


In [ ]:
# Country-level heat-flood scatter plot.

from matplotlib.gridspec import GridSpec
from scipy.stats import gaussian_kde

heat_flood = country_gdf[[
    "country_na", "HE_perpop", "sl_sevexp", "sl_pop", "EDI_qmean"
]].dropna(subset=["HE_perpop", "sl_sevexp"]).copy()
heat_flood = heat_flood[
    (heat_flood["HE_perpop"] > 0) & (heat_flood["sl_sevexp"] > 0)
].copy()

# Cap bubble scaling at the 95th percentile so a few large countries do not dominate.
pop_ref = heat_flood["sl_pop"].quantile(0.95)
heat_flood["bubble_size"] = (
    24 + 165 * np.sqrt(
        heat_flood["sl_pop"].clip(lower=0, upper=pop_ref) / pop_ref
    )
    if pop_ref > 0 else 60
)

fig = plt.figure(figsize=(4.4, 4), dpi=300, facecolor="white")

gs = GridSpec(
    2, 3,
    width_ratios=[4.0, 0.45, 0.18],
    height_ratios=[0.45, 4.0],
    wspace=0.035,
    hspace=0.035
)

ax_top = fig.add_subplot(gs[0, 0])
ax = fig.add_subplot(gs[1, 0], sharex=ax_top)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax)
cax = fig.add_subplot(gs[1, 2])

edi_vmin, edi_vmax = heat_flood["EDI_qmean"].quantile([0.02, 0.98])

scatter = ax.scatter(
    heat_flood["HE_perpop"], heat_flood["sl_sevexp"],
    s=heat_flood["bubble_size"],
    c=heat_flood["EDI_qmean"],
    cmap="coolwarm",
    vmin=edi_vmin,
    vmax=edi_vmax,
    alpha=0.86,
    edgecolor="white",
    linewidth=0.55,
    zorder=3,
)

# Use a log-scaled x-axis.
ax.set_xscale("log")
ax_top.set_xscale("log")

log_heat = np.log10(heat_flood["HE_perpop"])

slope, intercept = np.polyfit(log_heat, heat_flood["sl_sevexp"], 1)

x_line = np.geomspace(
    heat_flood["HE_perpop"].min(),
    heat_flood["HE_perpop"].max(),
    200
)

x_q25 = heat_flood["HE_perpop"].quantile(0.5)
y_q25 = heat_flood["sl_sevexp"].quantile(0.75)

ax.axvline(
    x_q25,
    color="#DB6E6E",
    linestyle=(0, (2, 2)),
    linewidth=0.9,
    alpha=0.85,
    zorder=2,
)

ax.axhline(
    y_q25,
    color="#DB6E6E",
    linestyle=(0, (2, 2)),
    linewidth=0.9,
    alpha=0.85,
    zorder=2,
)

x_log_grid = np.linspace(log_heat.min(), log_heat.max(), 300)
x_grid = 10 ** x_log_grid

kde_x = gaussian_kde(log_heat)
x_density = kde_x(x_log_grid)

ax_top.fill_between(
    x_grid,
    0,
    x_density,
    alpha=0.35,
    color="gray"
)

ax_top.plot(
    x_grid,
    x_density,
    color="black",
    linewidth=1.0
)

y_data = heat_flood["sl_sevexp"].values
y_grid = np.linspace(y_data.min(), y_data.max(), 300)

kde_y = gaussian_kde(y_data)
y_density = kde_y(y_grid)

ax_right.fill_betweenx(
    y_grid,
    0,
    y_density,
    alpha=0.35,
    color="gray"
)

ax_right.plot(
    y_density,
    y_grid,
    color="black",
    linewidth=1.0
)

from matplotlib.patches import Rectangle

x_min, x_max = ax.get_xlim()
y_min, y_max = ax.get_ylim()
x_split = np.clip(
    (np.log10(x_q25) - np.log10(x_min)) /
    (np.log10(x_max) - np.log10(x_min)),
    0, 1,
)
y_split = np.clip((y_q25 - y_min) / (y_max - y_min), 0, 1)

quadrants = [
    (0, 0, x_split, y_split, "#EAF1F5"),
    (x_split, 0, 1 - x_split, y_split, "#F5ECEA"),
    (0, y_split, x_split, 1 - y_split, "#F5ECEA"),
    (x_split, y_split, 1 - x_split, 1 - y_split, "#F7D1CC"),
]
for x0, y0, width, height, color in quadrants:
    ax.add_patch(Rectangle(
        (x0, y0), width, height,
        transform=ax.transAxes,
        facecolor=color,
        edgecolor="none",
        alpha=0.42,
        zorder=0,
        clip_on=True,
    ))

upper_right_countries = heat_flood.loc[
    (heat_flood["HE_perpop"] >= x_q25) &
    (heat_flood["sl_sevexp"] >= y_q25)
].sort_values("sl_sevexp", ascending=False).copy()

ax.set_xlabel(
    "SPW extreme heat exposure",
    fontsize=9.5,
    labelpad=7,
)

ax.set_ylabel(
    "SPW flood severity",
    fontsize=9.5,
    labelpad=7,
)

ax.tick_params(
    axis="both",
    which="major",
    labelsize=8.5,
    length=3,
    width=0.7,
    color="#444444",
)

ax.tick_params(
    axis="x",
    which="minor",
    labelsize=8.5,
    length=2,
    width=0.55,
)

ax_top.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    right=False,
    labelbottom=False,
    labeltop=False,
    labelleft=False,
    labelright=False,
)

ax_right.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    right=False,
    labelbottom=False,
    labeltop=False,
    labelleft=False,
    labelright=False,
)

ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#333333")
    spine.set_linewidth(0.75)

for spine in ax_top.spines.values():
    spine.set_visible(False)

for spine in ax_right.spines.values():
    spine.set_visible(False)

size_values = heat_flood["sl_pop"].quantile([0.25, 0.50, 0.75]).to_numpy()
size_handles = []
for value in size_values:
    marker_size = 24 + 165 * np.sqrt(min(value, pop_ref) / pop_ref)
    label = f"{value / 1e6:.1f} M" if value >= 1e6 else f"{value / 1e3:.0f} k"
    size_handles.append(ax.scatter(
        [], [], s=marker_size,
        facecolor="#7A7A7A", edgecolor="white",
        linewidth=0.5, alpha=0.82, label=label,
    ))

size_legend = ax.legend(
    handles=size_handles,
    title="Slum population",
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="none",
    framealpha=0.82,
    fontsize=7.5,
    title_fontsize=8,
    labelspacing=0.8,
    borderpad=0.6,
    handletextpad=0.7,
)
ax.add_artist(size_legend)

fig.subplots_adjust(left=0.12, right=0.92, bottom=0.13, top=0.91)

cax.set_axis_off()
cbar_ax = cax.inset_axes([0.25, 0.275, 0.50, 0.45])
cbar = fig.colorbar(
    scatter,
    cax=cbar_ax,
)

cbar.set_label("EDI", fontsize=8.5, labelpad=5)
cbar.ax.tick_params(labelsize=7.5, length=2, width=0.55)
cbar.outline.set_linewidth(0.55)
cbar.outline.set_edgecolor("#555555")

plt.savefig(
    "figure1c.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

from IPython.display import display
display(
    upper_right_countries[[
        "country_na", "HE_perpop", "sl_sevexp", "sl_pop", "EDI_qmean"
    ]].reset_index(drop=True)
)

## figure1f

In [ ]:
# Country ranking for incidence/infection and mortality/incidence.

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

def build_country_level_metrics():
    """Build the country-level metric table used by panels c, f, g, and h."""
    shp_path = r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
    country_shp_path = r"../data/country/country_SSA_HE2.shp"

    adm2 = gpd.read_file(shp_path).replace([np.inf, -np.inf], np.nan)
    country_boundary = gpd.read_file(country_shp_path)

    adm2["country_id"] = (
        pd.to_numeric(adm2["adm2ID"], errors="coerce") // 10000
    ).astype("Int64").astype(str)
    country_boundary["country_id"] = country_boundary["country_id"].astype(str)

    metric = adm2.to_crs("EPSG:6933").copy()
    metric.geometry = metric.geometry.make_valid()
    metric["_area"] = metric.geometry.area

    def add_weighted_terms(frame, value, weight, prefix):
        valid = frame[value].notna() & frame[weight].notna() & (frame[weight] > 0)
        frame[f"_{prefix}_num"] = frame[value].where(valid, 0) * frame[weight].where(valid, 0)
        frame[f"_{prefix}_den"] = frame[weight].where(valid, 0)

    add_weighted_terms(metric, "EDI_qmean", "_area", "edi")
    add_weighted_terms(metric, "HE_perpop", "sl_pop", "heat")
    add_weighted_terms(metric, "sl_sevexp", "sl_pop", "flood")

    sum_fields = [
        "_edi_num", "_edi_den", "_heat_num", "_heat_den",
        "_flood_num", "_flood_den", "sl_pop", "sl_pfpop",
        "sl_pfinf", "sl_pfinc", "sl_pfmort",
    ]
    country_stats = metric.groupby("country_id")[sum_fields].sum(min_count=1)
    country_stats["EDI_qmean"] = country_stats["_edi_num"] / country_stats["_edi_den"]
    country_stats["HE_perpop"] = country_stats["_heat_num"] / country_stats["_heat_den"]
    country_stats["sl_sevexp"] = country_stats["_flood_num"] / country_stats["_flood_den"]
    country_stats["sl_pfpr"] = country_stats["sl_pfinf"] / country_stats["sl_pfpop"]
    country_stats["inc_inf_ratio"] = country_stats["sl_pfinc"] / country_stats["sl_pfinf"]
    country_stats["mort_inc_ratio"] = country_stats["sl_pfmort"] / country_stats["sl_pfinc"]
    country_stats = country_stats.replace([np.inf, -np.inf], np.nan)

    country_gdf = country_boundary[["country_id", "country_na", "region", "geometry"]].merge(
        country_stats.reset_index(), on="country_id", how="left", validate="one_to_one"
    )
    if country_gdf.crs is not None and country_gdf.crs.to_epsg() != 4326:
        country_gdf = country_gdf.to_crs(4326)

    return adm2, country_gdf

required_country_columns = {
    "country_na", "EDI_qmean", "HE_perpop", "sl_sevexp",
    "sl_pfinf", "sl_pfinc", "sl_pfmort",
    "inc_inf_ratio", "mort_inc_ratio",
}
if "country_gdf" not in globals() or not required_country_columns.issubset(country_gdf.columns):
    adm2, country_gdf = build_country_level_metrics()

ranking_specs = [
    ("inc_inf_ratio", "Incidence / infection", "#E76F51"),
    ("mort_inc_ratio", "Mortality / incidence", "#6A51A3"),
]

# Compare the union of the top seven countries from both rankings.
rank_data = country_gdf[[
    "country_na", "inc_inf_ratio", "mort_inc_ratio"
]].dropna(subset=["inc_inf_ratio", "mort_inc_ratio"]).copy()
top_inc = set(rank_data.nlargest(7, "inc_inf_ratio")["country_na"])
top_mort = set(rank_data.nlargest(7, "mort_inc_ratio")["country_na"])
ranked = rank_data[rank_data["country_na"].isin(top_inc | top_mort)].copy()

# Sort by the mean percentile rank so countries high on both metrics appear near the top.
ranked["composite_rank"] = ranked[[
    "inc_inf_ratio", "mort_inc_ratio"
]].rank(pct=True).mean(axis=1)
ranked = ranked.sort_values("composite_rank").reset_index(drop=True)
y = np.arange(len(ranked))

fig, ax_inc = plt.subplots(figsize=(4.5, 6.6), dpi=300)
ax_mort = ax_inc.twiny()

for i in range(len(ranked)):
    if i % 2 == 0:
        ax_inc.axhspan(i - 0.5, i + 0.5, color="#F2F2F2", zorder=0)

offset = 0.16
ax_inc.hlines(
    y - offset, 0, ranked["inc_inf_ratio"],
    color="#F4B3A7", linewidth=1.1, zorder=1,
)
inc_points = ax_inc.scatter(
    ranked["inc_inf_ratio"], y - offset, s=28,
    color="#E76F51", edgecolor="white", linewidth=0.4,
    label="Incidence / infection", zorder=3,
)
ax_mort.hlines(
    y + offset, 0, ranked["mort_inc_ratio"],
    color="#C7B9E2", linewidth=1.1, zorder=1,
)
mort_points = ax_mort.scatter(
    ranked["mort_inc_ratio"], y + offset, s=28,
    color="#6A51A3", edgecolor="white", linewidth=0.4,
    label="Mortality / incidence", zorder=3,
)

ax_inc.set_yticks(y, ranked["country_na"], fontsize=9)
ax_inc.set_ylim(-0.7, len(ranked) - 0.3)
ax_inc.set_xlim(0, ranked["inc_inf_ratio"].max() * 1.08)
ax_mort.set_xlim(0, ranked["mort_inc_ratio"].max() * 1.08)
ax_inc.set_xlabel("Infection-to-incidence burden ratio", color="#E76F51", labelpad=7, fontsize=9)
ax_mort.set_xlabel("Incidence-to-mortality burden ratio", color="#6A51A3", labelpad=7, fontsize=9)
ax_inc.tick_params(axis="x", colors="#E76F51")
ax_mort.tick_params(axis="x", colors="#6A51A3")
ax_inc.tick_params(axis="y", length=0)
ax_inc.grid(False)
for spine in ax_inc.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(0.8)
ax_mort.spines[["right", "left", "bottom"]].set_visible(False)
ax_mort.spines["top"].set_visible(True)
ax_mort.spines["top"].set_color("black")
ax_mort.spines["top"].set_linewidth(0.8)

ax_inc.set_title(
    "Top countries by burden-transition ratios",
    loc="left", fontsize=10, pad=10,
)
plt.subplots_adjust(left=0.23, right=0.96, bottom=0.10, top=0.88)
plt.savefig(
    "figure1f.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

## figure1g

In [ ]:
# Figure 1g: wide grouped burden bars with environmental-risk matrix.
# This keeps the original figure1g aspect ratio while avoiding stacked shares.

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from IPython.display import display

plt.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

required_country_columns = {
    "country_id", "country_na", "EDI_qmean", "HE_perpop", "sl_sevexp",
    "sl_pfinf", "sl_pfinc", "sl_pfmort",
}
needs_country_rebuild = (
    "country_gdf" not in globals()
    or not required_country_columns.issubset(country_gdf.columns)
)
needs_adm2_rebuild = (
    "adm2" not in globals()
    or not {"country_id", "iso3"}.issubset(adm2.columns)
)

if needs_country_rebuild or needs_adm2_rebuild:
    if "build_country_level_metrics" in globals():
        adm2, country_gdf = build_country_level_metrics()
    else:
        shp_path = r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
        country_shp_path = r"../data/country/country_SSA_HE2.shp"

        adm2 = gpd.read_file(shp_path).replace([np.inf, -np.inf], np.nan)
        country_boundary = gpd.read_file(country_shp_path)

        adm2["country_id"] = (
            pd.to_numeric(adm2["adm2ID"], errors="coerce") // 10000
        ).astype("Int64").astype(str)
        country_boundary["country_id"] = country_boundary["country_id"].astype(str)

        metric = adm2.to_crs("EPSG:6933").copy()
        metric.geometry = metric.geometry.make_valid()
        metric["_area"] = metric.geometry.area

        def add_weighted_terms(frame, value, weight, prefix):
            valid = frame[value].notna() & frame[weight].notna() & (frame[weight] > 0)
            frame[f"_{prefix}_num"] = frame[value].where(valid, 0) * frame[weight].where(valid, 0)
            frame[f"_{prefix}_den"] = frame[weight].where(valid, 0)

        add_weighted_terms(metric, "EDI_qmean", "_area", "edi")
        add_weighted_terms(metric, "HE_perpop", "sl_pop", "heat")
        add_weighted_terms(metric, "sl_sevexp", "sl_pop", "flood")

        sum_fields = [
            "_edi_num", "_edi_den", "_heat_num", "_heat_den",
            "_flood_num", "_flood_den", "sl_pop", "sl_pfpop",
            "sl_pfinf", "sl_pfinc", "sl_pfmort",
        ]
        country_stats = metric.groupby("country_id")[sum_fields].sum(min_count=1)
        country_stats["EDI_qmean"] = country_stats["_edi_num"] / country_stats["_edi_den"]
        country_stats["HE_perpop"] = country_stats["_heat_num"] / country_stats["_heat_den"]
        country_stats["sl_sevexp"] = country_stats["_flood_num"] / country_stats["_flood_den"]
        country_stats = country_stats.replace([np.inf, -np.inf], np.nan)

        country_gdf = country_boundary[["country_id", "country_na", "region", "geometry"]].merge(
            country_stats.reset_index(), on="country_id", how="left", validate="one_to_one"
        )
        if country_gdf.crs is not None and country_gdf.crs.to_epsg() != 4326:
            country_gdf = country_gdf.to_crs(4326)

lollipop_data = country_gdf.copy()
lollipop_data["country_id"] = lollipop_data["country_id"].astype(str)
adm2["country_id"] = adm2["country_id"].astype(str)

iso_lookup = adm2[["country_id", "iso3"]].dropna().drop_duplicates("country_id")
lollipop_data = lollipop_data.merge(iso_lookup, on="country_id", how="left")
lollipop_data["plot_label"] = lollipop_data["iso3"].fillna(
    lollipop_data["country_na"].str[:3].str.upper()
)

risk_specs = [
    ("EDI_qmean", "EDI"),
    ("HE_perpop", "Heat"),
    ("sl_sevexp", "Flood"),
]
for field, label in risk_specs:
    threshold = country_gdf[field].median(skipna=True)
    lollipop_data[f"high_{label.lower()}"] = lollipop_data[field] > threshold
risk_columns = ["high_edi", "high_heat", "high_flood"]
lollipop_data["risk_count"] = lollipop_data[risk_columns].sum(axis=1)

burden_specs = [
    ("sl_pfinf", "Infections", "#80B1D3", -0.26),
    ("sl_pfinc", "Incidence", "#FDB462", 0.00),
    ("sl_pfmort", "Mortality", "#FB8072", 0.26),
]
burden_share_fields = []
for field, label, color, offset in burden_specs:
    share_field = f"{field}_share"
    total = lollipop_data[field].sum(skipna=True)
    lollipop_data[share_field] = np.where(
        total > 0,
        100 * lollipop_data[field].fillna(0) / total,
        np.nan,
    )
    burden_share_fields.append(share_field)

lollipop_data["max_outcome_share"] = lollipop_data[burden_share_fields].max(axis=1)
lollipop_data = (
    lollipop_data.dropna(subset=["max_outcome_share"])
    .sort_values(["max_outcome_share", "country_na"], ascending=[False, True])
    .head(30)
    .reset_index(drop=True)
)

x = np.arange(len(lollipop_data))
fig = plt.figure(figsize=(15, 4), dpi=300)
gs = GridSpec(2, 1, height_ratios=[4.2, 1.35], hspace=0.04, figure=fig)
ax_burden = fig.add_subplot(gs[0])
ax_dot = fig.add_subplot(gs[1], sharex=ax_burden)

bar_width = 0.22
for field, label, color, offset in burden_specs:
    share_field = f"{field}_share"
    values = lollipop_data[share_field].to_numpy()
    xpos = x + offset
    ax_burden.bar(
        xpos, values, width=bar_width, color=color, alpha=0.92,
        edgecolor="white", linewidth=0.45, label=label, zorder=2,
    )

ax_burden.set_ylabel("Outcome-specific share\nof SSA burden (%)", fontsize=10)
ax_burden.tick_params(axis="y", labelsize=10)
ax_burden.tick_params(axis="x", labelbottom=False)
ax_burden.set_xlim(-0.7, len(lollipop_data) - 0.3)
y_tick_step = 10
y_axis_top = np.ceil(lollipop_data[burden_share_fields].max().max() * 1.12 / y_tick_step) * y_tick_step
ax_burden.set_ylim(0, y_axis_top)
ax_burden.set_yticks(np.arange(0, y_axis_top + 0.1, y_tick_step))
ax_burden.spines[["top", "right"]].set_visible(False)
ax_burden.grid(axis="y", color="#E6E6E6", linewidth=0.55, zorder=0)
ax_burden.legend(
    frameon=False, ncol=3, loc="upper left",
    bbox_to_anchor=(0.16, 1.0), fontsize=11,
)

row_labels = [label for field, label in risk_specs]
row_y = np.array([2, 1, 0])
for y_value in row_y:
    ax_dot.scatter(x, np.full(len(x), y_value), s=28, color="#DDDDDD", zorder=1)

for i, row in lollipop_data.iterrows():
    active_y = row_y[row[risk_columns].to_numpy(dtype=bool)]
    if len(active_y) >= 2:
        ax_dot.plot(
            [i, i], [active_y.min(), active_y.max()],
            color="#333333", linewidth=1.0, zorder=2,
        )
    if len(active_y) > 0:
        ax_dot.scatter(
            np.full(len(active_y), i), active_y, s=34,
            color="#222222", edgecolor="white", linewidth=0.3, zorder=3,
        )

ax_dot.set_yticks(row_y, row_labels, fontsize=10)
ax_dot.set_xticks(x, lollipop_data["plot_label"], rotation=90, fontsize=10)
ax_dot.set_xlim(-0.7, len(lollipop_data) - 0.3)
ax_dot.set_ylim(-0.55, 2.55)
ax_dot.set_xlabel("Country", fontsize=10)
ax_dot.tick_params(axis="y", length=0)
ax_dot.grid(axis="x", color="#F0F0F0", linewidth=0.4)
ax_dot.spines[["top", "right", "left"]].set_visible(False)

plt.subplots_adjust(left=0.08, right=0.99, top=0.92, bottom=0.17)
plt.savefig(
    "figure1g_lollipop.pdf",
    dpi=600,
    bbox_inches="tight",
)
plt.savefig(
    "figure1g_lollipop.png",
    dpi=600,
    bbox_inches="tight",
)
plt.savefig(
    "figure1g_lollipop.svg",
    bbox_inches="tight",
)
plt.savefig(
    "figure1g_lollipop.tiff",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

display(lollipop_data[[
    "country_na", "plot_label", "max_outcome_share", "risk_count",
    "sl_pfinf_share", "sl_pfinc_share", "sl_pfmort_share",
    "high_edi", "high_heat", "high_flood",
]])


## figure1h

In [ ]:
# Circular overlap analysis for the three elevated environmental risks.
# Use each indicator's median among complete-case countries as the threshold.

from matplotlib.patches import Patch
from IPython.display import display

risk_overlap = country_gdf[[
    "country_na", "EDI_qmean", "HE_perpop", "sl_sevexp"
]].dropna().copy()

risk_overlap["high_edi"] = (
    risk_overlap["EDI_qmean"] > risk_overlap["EDI_qmean"].median()
)
risk_overlap["high_heat"] = (
    risk_overlap["HE_perpop"] > risk_overlap["HE_perpop"].median()
)
risk_overlap["high_flood"] = (
    risk_overlap["sl_sevexp"] > risk_overlap["sl_sevexp"].median()
)

combination_specs = [
    ("EDI only", True, False, False),
    ("Heat only", False, True, False),
    ("Flood only", False, False, True),
    ("EDI + Heat", True, True, False),
    ("EDI + Flood", True, False, True),
    ("Heat + Flood", False, True, True),
    ("EDI + Heat + Flood", True, True, True),
]

combination_counts = {}
combination_rows = []
for label, edi, heat, flood in combination_specs:
    mask = (
        (risk_overlap["high_edi"] == edi)
        & (risk_overlap["high_heat"] == heat)
        & (risk_overlap["high_flood"] == flood)
    )
    countries = risk_overlap.loc[mask, "country_na"].sort_values().tolist()
    combination_counts[label] = len(countries)
    combination_rows.append({
        "Risk combination": label,
        "Number of countries": len(countries),
        "Countries": ", ".join(countries),
    })

risk_combination_table = pd.DataFrame(combination_rows)
risk_combination_table["Risk number"] = [1, 1, 1, 2, 2, 2, 3]
labels = risk_combination_table["Risk combination"].to_numpy()
counts = risk_combination_table["Number of countries"].to_numpy()
risk_numbers = risk_combination_table["Risk number"].to_numpy()
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False)
bar_width = 2 * np.pi / len(labels) * 0.76
inner_radius = max(2.0, counts.max() * 0.28)
# Low-saturation colors transition from cool to warm as the number of risks increases.
risk_colors = {
    1: "#78C6C0",  # light teal
    2: "#B9A4E3",  # light violet
    3: "#F29A8B",  # light coral
}
bar_colors = [risk_colors[number] for number in risk_numbers]

fig_circular, ax_circular = plt.subplots(
    figsize=(7.0, 7.0), dpi=300,
    subplot_kw={"projection": "polar"},
)
ax_circular.bar(
    angles, counts,
    width=bar_width,
    bottom=inner_radius,
    color=bar_colors,
    edgecolor="white",
    linewidth=1.0,
    alpha=0.9,
)

for angle, count, label in zip(angles, counts, labels):
    ax_circular.text(
        angle, inner_radius + count / 2,
        str(count),
        ha="center", va="center",
        fontsize=9, fontweight="semibold",
        color="#222222",
    )

ax_circular.set_theta_offset(np.pi / 2)
ax_circular.set_theta_direction(-1)
ax_circular.set_xticks([])
ax_circular.set_yticklabels([])
ax_circular.grid(axis="y", color="#ffffff", linewidth=0.55)
ax_circular.spines["polar"].set_visible(False)
ax_circular.set_ylim(0, inner_radius + counts.max() * 1.32)
ax_circular.text(
    0.5, 0.5,
    f"Risk combination",
    transform=ax_circular.transAxes,
    ha="center", va="center",
    fontsize=9, color="#000000",
)
ax_circular.legend(
    handles=[
        Patch(facecolor=risk_colors[1], label="One elevated risk"),
        Patch(facecolor=risk_colors[2], label="Two elevated risks"),
        Patch(facecolor=risk_colors[3], label="Three elevated risks"),
    ],
    frameon=False, loc="lower center",
    bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=8,
)
fig_circular.tight_layout()
fig_circular.savefig(
    "figure1h.pdf",
    dpi=600, bbox_inches="tight",
)
plt.show()
display(risk_combination_table)